In [0]:
%sql
USE CATALOG retail_dwh;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_dwh.silver.customers
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/silver/customers'
AS
WITH ranked AS (
    SELECT
        TRIM(CustomerID)                    AS CustomerID,
        INITCAP(TRIM(CustomerName))         AS CustomerName,
        LOWER(TRIM(Email))                  AS Email,
        TRIM(City)                          AS City,
        TRIM(Address)                       AS Address,
        CAST(LastUpdated AS DATE)           AS LastUpdated,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(CustomerID)
            ORDER BY LastUpdated DESC
        )                                   AS rn
    FROM retail_dwh.bronze.customers
    WHERE CustomerID IS NOT NULL
      AND CustomerName IS NOT NULL
      AND Email IS NOT NULL
)
SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    LastUpdated,
    current_timestamp()                     AS silver_timestamp
FROM ranked
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_dwh.silver.products
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/silver/products'
AS
SELECT DISTINCT
    TRIM(ProductID)                         AS ProductID,
    INITCAP(TRIM(ProductName))              AS ProductName,
    INITCAP(TRIM(Category))                 AS Category,
    CAST(UnitPrice AS DECIMAL(10,2))        AS UnitPrice,
    current_timestamp()                     AS silver_timestamp
FROM retail_dwh.bronze.products
WHERE ProductID IS NOT NULL
  AND ProductName IS NOT NULL
  AND CAST(UnitPrice AS DECIMAL(10,2)) > 0;

In [0]:
%sql
CREATE OR REPLACE TABLE retail_dwh.silver.sales_transactions
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/silver/sales_transactions'
AS
WITH deduped AS (
    SELECT
        TRIM(TransactionID)                 AS TransactionID,
        TRIM(CustomerID)                    AS CustomerID,
        TRIM(ProductID)                     AS ProductID,
        TRIM(StoreID)                       AS StoreID,
        CAST(Quantity AS INT)               AS Quantity,
        TO_DATE(TxnDate)                    AS TxnDate,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(TransactionID)
            ORDER BY TxnDate
        )                                   AS rn
    FROM retail_dwh.bronze.sales_transactions
    WHERE TransactionID IS NOT NULL
      AND CustomerID IS NOT NULL
      AND CAST(Quantity AS INT) > 0
      AND TxnDate IS NOT NULL
)
SELECT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    DATE_FORMAT(TxnDate, 'yyyy-MM-dd')      AS TxnDate,
    current_timestamp()                     AS silver_timestamp
FROM deduped
WHERE rn = 1
  AND CustomerID IN (
      SELECT CustomerID
      FROM retail_dwh.silver.customers
  );

In [0]:
%sql
CREATE OR REPLACE TABLE retail_dwh.silver.stores
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/silver/stores'
AS
SELECT DISTINCT
    TRIM(StoreID)                           AS StoreID,
    INITCAP(TRIM(StoreName))                AS StoreName,
    INITCAP(TRIM(Region))                   AS Region,
    current_timestamp()                     AS silver_timestamp
FROM retail_dwh.bronze.stores
WHERE StoreID IS NOT NULL
  AND StoreName IS NOT NULL
  AND Region IS NOT NULL
  AND TRIM(Region) != '';